# 03 Co-Citation Network

## Setup

In [2]:
import gc
import time
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
from datasketch import MinHash, MinHashLSH
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)
pd.set_option('display.max_rows', 100)
warnings.filterwarnings('ignore', category=FutureWarning)
cwd = Path.cwd()
PROJECT_ROOT = cwd.parent if cwd.name == 'notebooks' else cwd
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
DIRECT_DATA_DIR = PROJECT_ROOT / 'outputs' / '02_direct_citation_network' / 'data'
NOTEBOOK_OUTPUT_DIR = PROJECT_ROOT / 'outputs' / '03_cocitation_network'
FIGURE_DIR = NOTEBOOK_OUTPUT_DIR / 'figures'
TABLE_DIR = NOTEBOOK_OUTPUT_DIR / 'tables'
DATA_OUTPUT_DIR = NOTEBOOK_OUTPUT_DIR / 'data'
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)
DATA_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## 1. Co-Citation Parameters

In [3]:
MIN_CITING = 5
NUM_PERM = 128
LSH_THRESHOLD = 0.2
MIN_SHARED_CITERS = 5
TOP_K = 30
MINHASH_SEED = 1
print('Co-citation parameters')
print('----------------------')
print(f'Minimum citing papers: {MIN_CITING}')
print(f'MinHash permutations: {NUM_PERM}')
print(f'LSH threshold: {LSH_THRESHOLD}')
print(f'Minimum shared citers: {MIN_SHARED_CITERS}')
print(f'Top K neighbors: {TOP_K}')
print(f'MinHash seed: {MINHASH_SEED}')

Co-citation parameters
----------------------
Minimum citing papers: 5
MinHash permutations: 128
LSH threshold: 0.2
Minimum shared citers: 5
Top K neighbors: 30
MinHash seed: 1


## Load Final Citation Graph

In [4]:
citation_edges_raw = pd.read_pickle(PROCESSED_DIR / 'citation_edges_final.pkl')
papers = pd.read_feather(DIRECT_DATA_DIR / 'papers_with_degrees.feather')
print('Final citation edges:', citation_edges_raw.shape)
print('Papers:', papers.shape)

Final citation edges: (18904534, 2)
Papers: (3026412, 32)


## Standardize Edge Columns

In [5]:
citation_edges = citation_edges_raw.rename(columns={'citing_paper_id': 'citing', 'cited_paper_id': 'cited'})[['citing', 'cited']].copy()
del citation_edges_raw
gc.collect()
papers['id'] = papers['id'].astype(str)
citation_edges['citing'] = citation_edges['citing'].astype(str)
citation_edges['cited'] = citation_edges['cited'].astype(str)
citation_edges = citation_edges.reset_index(drop=True)
n_cs = int(papers['primary_field'].eq('Computer Science').sum())
print(f'Final papers: {len(papers):,}')
print(f'Final filtered direct edges: {len(citation_edges):,}')
print(f'Computer Science papers: {n_cs:,}')

Final papers: 3,026,412
Final filtered direct edges: 18,904,534
Computer Science papers: 1,542,691


## 4. Identify Papers with Sufficient Citer Sets

In [6]:
citer_counts = citation_edges.groupby('cited', sort=False).size().rename('n_citers')
eligible_citer_counts = citer_counts[citer_counts >= MIN_CITING].copy()
eligible_papers = set(eligible_citer_counts.index)
print(f'Papers with at least one citer: {len(citer_counts):,}')
print(f'Papers with at least {MIN_CITING} citers: {len(eligible_citer_counts):,}')
print(f'Share of final paper universe eligible: {100 * len(eligible_citer_counts) / len(papers):.2f}%')
display(eligible_citer_counts.describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).to_frame('n_citers').round(2))

Papers with at least one citer: 1,915,670
Papers with at least 5 citers: 711,808
Share of final paper universe eligible: 23.52%


,n_citers
count,711808.00
mean,23.31
std,109.91
min,5.00
25%,7.00
50%,10.00
75%,20.00
90%,42.00
95%,69.00
99%,207.93


## 5. Build Exact Final Citer Sets

In [7]:
eligible_edges = citation_edges[citation_edges['cited'].isin(eligible_papers)][['cited', 'citing']].copy()
t0 = time.time()
citer_sets = eligible_edges.groupby('cited', sort=False)['citing'].agg(lambda x: frozenset(x)).to_dict()
elapsed = time.time() - t0
print(f'Built {len(citer_sets):,} exact citer sets in {elapsed / 60:.2f} minutes.')
del eligible_edges
gc.collect()

Built 711,808 exact citer sets in 0.21 minutes.


0

## 6. Build MinHash Signatures

In [8]:
def build_minhash(items):
    mh = MinHash(num_perm=NUM_PERM, seed=MINHASH_SEED)
    for item in items:
        mh.update(str(item).encode('utf-8'))
    return mh
t0 = time.time()
minhashes = {}
for idx, (paper_id, citers) in enumerate(citer_sets.items(), start=1):
    minhashes[paper_id] = build_minhash(citers)
elapsed = time.time() - t0
print(f'Finished {len(minhashes):,} MinHash signatures in {elapsed / 60:.2f} minutes.')

Finished 711,808 MinHash signatures in 3.29 minutes.


## 7. Build the MinHash LSH Index

In [9]:
lsh = MinHashLSH(threshold=LSH_THRESHOLD, num_perm=NUM_PERM)
t0 = time.time()
for idx, (paper_id, mh) in enumerate(minhashes.items(), start=1):
    lsh.insert(paper_id, mh)
elapsed = time.time() - t0
print(f'LSH index built in {elapsed / 60:.2f} minutes.')

LSH index built in 0.71 minutes.


## 8. Generate Candidate Neighbors and Compute Exact Co-Citation Strength

In [10]:
neighbor_rows = []
candidate_count_total = 0
valid_exact_count_total = 0
t0 = time.time()
n_sources = len(minhashes)
for idx, paper_id in enumerate(minhashes.keys(), start=1):
    source_set = citer_sets[paper_id]
    candidates = lsh.query(minhashes[paper_id])
    candidates = [candidate_id for candidate_id in candidates if candidate_id != paper_id]
    candidate_count_total += len(candidates)
    scored = []
    for neighbor_id in candidates:
        target_set = citer_sets[neighbor_id]
        shared = len(source_set.intersection(target_set))
        if shared < MIN_SHARED_CITERS:
            continue
        union_size = len(source_set) + len(target_set) - shared
        jaccard = shared / union_size if union_size > 0 else 0.0
        scored.append((neighbor_id, shared, jaccard))
    valid_exact_count_total += len(scored)
    scored.sort(key=lambda row: (-row[1], -row[2], row[0]))
    for rank, (neighbor_id, shared, jaccard) in enumerate(scored[:TOP_K], start=1):
        neighbor_rows.append({'paper_id': paper_id, 'neighbor_id': neighbor_id, 'shared_citers': shared, 'jaccard': jaccard, 'neighbor_rank': rank})
elapsed = time.time() - t0
cocitation_neighbors = pd.DataFrame(neighbor_rows)
print()
print('Candidate search completed.')
print(f'Total LSH candidate occurrences: {candidate_count_total:,}')
print(f'Exact valid candidate occurrences (shared citers >= {MIN_SHARED_CITERS}): {valid_exact_count_total:,}')
print(f'Directed top-K neighborhood rows: {len(cocitation_neighbors):,}')
print(f'Runtime: {elapsed / 60:.2f} minutes')


Candidate search completed.
Total LSH candidate occurrences: 3,256,966
Exact valid candidate occurrences (shared citers >= 5): 464,130
Directed top-K neighborhood rows: 464,130
Runtime: 0.44 minutes


## 10. Construct the Undirected Weighted Co-Citation Graph

In [11]:
source = cocitation_neighbors['paper_id'].astype(str)
target = cocitation_neighbors['neighbor_id'].astype(str)
pair_u = np.where(source.to_numpy() < target.to_numpy(), source.to_numpy(), target.to_numpy())
pair_v = np.where(source.to_numpy() < target.to_numpy(), target.to_numpy(), source.to_numpy())
undirected = cocitation_neighbors[['shared_citers', 'jaccard']].copy()
undirected['paper_a'] = pair_u
undirected['paper_b'] = pair_v
cocitation_edges = undirected.groupby(['paper_a', 'paper_b'], as_index=False, sort=False).agg(shared_citers=('shared_citers', 'max'), jaccard=('jaccard', 'max'))
print(f'Undirected retained co-citation edges: {len(cocitation_edges):,}')
del (source, target, pair_u, pair_v, undirected)
gc.collect()

Undirected retained co-citation edges: 232,065


0

## 11. Co-Citation Edge-Weight Summary

In [12]:
cocitation_weight_summary = pd.DataFrame({'shared_citers': cocitation_edges['shared_citers'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]), 'jaccard': cocitation_edges['jaccard'].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99])})
display(cocitation_weight_summary.round(4))
cocitation_weight_summary.to_csv(TABLE_DIR / 'cocitation_edge_weight_summary_final.csv')

,shared_citers,jaccard
count,232065.0000,232065.0000
mean,13.2727,0.1780
std,33.6626,0.1252
min,5.0000,0.0012
25%,6.0000,0.0921
50%,8.0000,0.1481
75%,12.0000,0.2273
90%,23.0000,0.3333
95%,36.0000,0.4211
99%,94.0000,0.6250


## 12. Construct Co-Citation Node Features

In [13]:
left = cocitation_edges.rename(columns={'paper_a': 'paper_id', 'paper_b': 'neighbor_id'})[['paper_id', 'neighbor_id', 'shared_citers', 'jaccard']]
right = cocitation_edges.rename(columns={'paper_b': 'paper_id', 'paper_a': 'neighbor_id'})[['paper_id', 'neighbor_id', 'shared_citers', 'jaccard']]
incidence = pd.concat([left, right], ignore_index=True)
node_features_observed = incidence.groupby('paper_id').agg(cocitation_degree=('neighbor_id', 'nunique'), cocitation_strength=('shared_citers', 'sum'), cocitation_mean_shared_citers=('shared_citers', 'mean'), cocitation_max_shared_citers=('shared_citers', 'max'), cocitation_mean_jaccard=('jaccard', 'mean'), cocitation_max_jaccard=('jaccard', 'max'))
cocitation_node_features = pd.DataFrame({'id': papers['id'].astype(str)})
cocitation_node_features = cocitation_node_features.merge(node_features_observed, how='left', left_on='id', right_index=True)
count_cols = ['cocitation_degree', 'cocitation_strength']
for col in count_cols:
    cocitation_node_features[col] = cocitation_node_features[col].fillna(0)
other_cols = ['cocitation_mean_shared_citers', 'cocitation_max_shared_citers', 'cocitation_mean_jaccard', 'cocitation_max_jaccard']
for col in other_cols:
    cocitation_node_features[col] = cocitation_node_features[col].fillna(0.0)
cocitation_node_features['has_cocitation_neighbor'] = cocitation_node_features['cocitation_degree'] > 0
display(cocitation_node_features[['cocitation_degree', 'cocitation_strength', 'cocitation_mean_shared_citers', 'cocitation_mean_jaccard']].describe(percentiles=[0.25, 0.5, 0.75, 0.9, 0.95, 0.99]).round(3))
del left, right, incidence, node_features_observed
gc.collect()

,cocitation_degree,cocitation_strength,cocitation_mean_shared_citers,cocitation_mean_jaccard
count,3026412.000,3026412.000,3026412.000,3026412.000
mean,0.153,2.035,0.679,0.012
std,0.828,24.845,8.244,0.057
min,0.000,0.000,0.000,0.000
25%,0.000,0.000,0.000,0.000
50%,0.000,0.000,0.000,0.000
75%,0.000,0.000,0.000,0.000
90%,0.000,0.000,0.000,0.000
95%,1.000,6.000,5.000,0.093
99%,4.000,45.000,13.000,0.294


0

## 13. Network Coverage Summary

In [14]:
n_final = len(papers)
n_eligible = len(citer_sets)
n_with_topk = cocitation_neighbors['paper_id'].nunique()
n_with_undirected_neighbor = int(cocitation_node_features['has_cocitation_neighbor'].sum())
coverage_summary = pd.DataFrame({'Statistic': ['Final analytical papers', f'Papers with at least {MIN_CITING} citers', 'Papers with at least one directed retained neighbor', 'Papers with at least one undirected retained neighbor', 'Directed top-K neighborhood rows', 'Undirected retained edges'], 'Value': [n_final, n_eligible, n_with_topk, n_with_undirected_neighbor, len(cocitation_neighbors), len(cocitation_edges)]})
display(coverage_summary)
coverage_summary.to_csv(TABLE_DIR / 'cocitation_network_coverage_final.csv', index=False)

,Statistic,Value
0,Final analytical papers,3026412
1,Papers with at least 5 citers,711808
2,Papers with at least one directed retained nei...,188119
3,Papers with at least one undirected retained n...,188119
4,Directed top-K neighborhood rows,464130
5,Undirected retained edges,232065


## Computer Science Coverage

In [15]:
cs_ids = set(papers.loc[papers['primary_field'].eq('Computer Science'), 'id'].astype(str))
cs_features = cocitation_node_features[cocitation_node_features['id'].isin(cs_ids)]
cs_summary = pd.Series({'n_cs_papers': len(cs_features), 'n_cs_with_cocitation_neighbor': int(cs_features['has_cocitation_neighbor'].sum()), 'coverage_pct': 100 * cs_features['has_cocitation_neighbor'].mean(), 'mean_cocitation_degree': cs_features['cocitation_degree'].mean(), 'median_cocitation_degree': cs_features['cocitation_degree'].median(), 'mean_cocitation_strength': cs_features['cocitation_strength'].mean()})
display(cs_summary.to_frame('Value'))
cs_summary.to_csv(TABLE_DIR / 'cocitation_cs_coverage_final.csv')

,Value
n_cs_papers,1.542691e+06
n_cs_with_cocitation_neighbor,1.165790e+05
coverage_pct,7.556860e+00
mean_cocitation_degree,1.946540e-01
median_cocitation_degree,0.000000e+00
mean_cocitation_strength,2.733308e+00


## Save Co-Citation Artifacts

In [16]:
neighbors_path = DATA_OUTPUT_DIR / 'cocitation_neighbors_topk.feather'
edges_path_out = DATA_OUTPUT_DIR / 'cocitation_edges.feather'
features_path = DATA_OUTPUT_DIR / 'cocitation_node_features.feather'
cocitation_neighbors.reset_index(drop=True).to_feather(neighbors_path)
cocitation_edges.reset_index(drop=True).to_feather(edges_path_out)
cocitation_node_features.reset_index(drop=True).to_feather(features_path)
print('Saved:', DATA_OUTPUT_DIR)

Saved: c:\Users\blaze\OneDrive\Documents\GitHub\citation-gender-imbalance\outputs\03_cocitation_network\data
